In [125]:
%load_ext autoreload
%autoreload 2

from Utils import Notebook
from Utils import Tex
from IPython.display import Math, display
import numpy as np
import scipy.linalg as la

from IPython.display import display, Math, Latex,Markdown

import ControllerDesigner


Notebook.setup()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


LaTeX has been enabled for text rendering.


### Definição da Planta

In [126]:
def check_system_properties(A, B, C, tol=1e-9):
  n = A.shape[0]
  controllability = B.copy()
  for i in range(1, n):
    controllability = np.hstack(
        (controllability, np.linalg.matrix_power(A, i) @ B))
  rank_ctrb = np.linalg.matrix_rank(controllability, tol=tol)
  controllable = rank_ctrb == n

  observability = C.copy()
  for i in range(1, n):
    observability = np.vstack(
        (observability, C @ np.linalg.matrix_power(A, i)))
  rank_obsv = np.linalg.matrix_rank(observability, tol=tol)
  observable = rank_obsv == n

  eig_A = np.linalg.eigvals(A)
  stabilizable = True

  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.hstack((eig * np.eye(n) - A, B))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        stabilizable = False
        break

  detectable = True
  for eig in eig_A:
    if np.real(eig) >= -tol:
      pbh_matrix = np.vstack((eig * np.eye(n) - A, C))
      if np.linalg.matrix_rank(pbh_matrix, tol=tol) < n:
        detectable = False
        break

  return {
      "controllability_rank": rank_ctrb,
      "observability_rank": rank_obsv,
      "controllable": controllable,
      "observable": observable,
      "stabilizable": stabilizable,
      "detectable": detectable,
      "eigenvalues": eig_A,
  }


A = np.array([
    [0.0, 1.0],
    [-4.0, 0.4]
])

B = np.array([
    [0.0],
    [1.0]
])

C = np.array([
    [1.0, 0.0]
])

results = check_system_properties(A, B, C)

print("=" * 60)
print("SYSTEM PROPERTIES")
print("=" * 60)

print(f"Controllability rank : "
      f"{results['controllability_rank']} / {A.shape[0]}")

print(f"Observability rank   : "
      f"{results['observability_rank']} / {A.shape[0]}")

print(f"Controllable         : {results['controllable']}")
print(f"Observable           : {results['observable']}")
print(f"Stabilizable         : {results['stabilizable']}")
print(f"Detectable           : {results['detectable']}")

print("\nEigenvalues of A:")
for eig in results["eigenvalues"]:
  print(f"  {eig}")

SYSTEM PROPERTIES
Controllability rank : 2 / 2
Observability rank   : 2 / 2
Controllable         : True
Observable           : True
Stabilizable         : True
Detectable           : True

Eigenvalues of A:
  (0.20000000000000012+1.98997487421324j)
  (0.20000000000000012-1.98997487421324j)


### Co-projeto do Controlador baseado em Eventos

In [127]:
import numpy as np
from IPython.display import Math, display

# -------------------------------------------------------------------------
# 1. Parâmetros do Sistema e Projeto
# -------------------------------------------------------------------------
h = 1e-3
lambd = 1e-2

upsilon1 = 1e-2
upsilon2 = 1e-2
upsilon3 = 0.5
upsilon4 = 0.5

# Matrizes da planta
A = np.array([
    [0.0, 1.0],
    [3.75, 0.0]
], dtype=np.float64)

B = np.array([
    [0.0],
    [0.25]
], dtype=np.float64)

C = np.array([
    [0.1, 0.5]
], dtype=np.float64)

# Base ortogonal complementar para T = [C; T2]
T2 = np.array([
    [0.0, 1.0]
], dtype=np.float64)

ctrl_params = {'A': A, 'B': B, 'C': C, 'T2': T2, 'h': h, 'λ': lambd,
               'υ1': upsilon1, 'υ2': upsilon2, 'υ3': upsilon3, 'υ4': upsilon4, }

synth_res = ControllerDesigner.synthesize_output_based_setm(
    ctrl_params, eps=1e-6, verbose=False)

if synth_res is None:
  print("ERRO: Síntese Infeasible ou falha na recuperação das matrizes.")
else:
  # Ganho e Lyapunov
  K = synth_res['controller']['K']
  P = synth_res['functional']['P']
  R = synth_res['functional']['R']

  # ETM Sensor-Controlador (SC)
  Ψ_sc = synth_res['etm']['sc']['Ψ']
  Ξ_sc = synth_res['etm']['sc']['Ξ']

  # ETM Controlador-Atuador (CA)
  Ψ_ca = synth_res['etm']['ca']['Ψ']
  Ξ_ca = synth_res['etm']['ca']['Ξ']

  # Verificação rápida de estabilidade nominal contínua
  eig_cl = np.linalg.eigvals(A + B @ K)
  print("=== SÍNTESE CONCLUÍDA COM SUCESSO ===")
  print(f"Status do Solver: {synth_res['solver_status']}")
  print(f"Autovalores Nominais de Malha Fechada: {np.round(eig_cl, 4)}")

  display(Math(rf"""
    \begin{{aligned}}
        K &= {Tex.mat2tex(K)}, \qquad 
        P = {Tex.mat2tex(P)}, \qquad 
        R = {Tex.mat2tex(R)} \\[10pt]
        \Psi_{{\mathrm{{sc}}}} &= {Tex.mat2tex(Ψ_sc)}, \qquad 
        \Xi_{{\mathrm{{sc}}}} = {Tex.mat2tex(Ξ_sc)} \\[10pt]
        \Psi_{{\mathrm{{ca}}}} &= {Tex.mat2tex(Ψ_ca)}, \qquad 
        \Xi_{{\mathrm{{ca}}}} = {Tex.mat2tex(Ξ_ca)}
    \end{{aligned}}
    """))

=== SÍNTESE CONCLUÍDA COM SUCESSO ===
Status do Solver: optimal
Autovalores Nominais de Malha Fechada: [-0.6431 -3.9713]


<IPython.core.display.Math object>

### Projeto do Observador Impulsivo

In [128]:
import numpy as np


def compute_nu_max_from_multiplier(A: np.ndarray, M: float) -> float:
  """
  Calcula o intervalo máximo entre eventos (nu_max) a partir do multiplicador 
  do envelope de fluxo M = exp(mu_2(A) * nu_max).

  Parâmetros:
  - A: Matriz dinâmica contínua da planta (numpy.ndarray, formato nx x nx)
  - M: Valor do multiplicador do fluxo (float, M > 0)

  Retorna:
  - nu_max: Intervalo máximo entre eventos correspondente (float)
  """
  if M <= 0:
    raise ValueError("O multiplicador M deve ser estritamente positivo.")

  A_sym = (A + A.T) / 2.0
  mu_2_A = float(np.max(np.linalg.eigvalsh(A_sym)))

  if abs(mu_2_A) < 1e-12:
    if abs(M - 1.0) < 1e-6:
      raise ValueError(
          "Com mu_2(A) = 0, M = 1.0 é satisfeito para qualquer nu_max "
          "(o intervalo é indeterminado/livre)."
      )
    else:
      raise ValueError(
          f"Incompatibilidade: mu_2(A) ≈ 0 não permite um multiplicador M = {M} != 1.0."
      )

  if mu_2_A < 0 and M > 1.0:
    raise ValueError(
        f"Para sistemas com medida matricial negativa (mu_2(A) = {mu_2_A:.4f}), "
        f"o fluxo decresce (M <= 1.0). Um multiplicador M = {M} é fisicamente inexequível."
    )

  nu_max = np.log(M) / mu_2_A

  if nu_max < 0:
    raise ValueError("O valor calculado de nu_max resultou negativo.")

  return float(nu_max)


M_target = 1.5
try:
  nu_bar = compute_nu_max_from_multiplier(A, M_target)
  print(f"Intervalo máximo calculado é: nu_max = {nu_bar:.4f} s")
except ValueError as e:
  print(f"[Erro de Validação] {e}")

Intervalo máximo calculado é: nu_max = 0.1707 s


In [129]:
# 1. Síntese Sistemática dos Ganhos Contínuos (Etapa 1)
cont_design = ControllerDesigner.synthesize_continuous_gains(
    A=A,
    C=C,
    alpha_0=2.1,
    alpha_z=3,
    g0_method="analytical",
    verbose=False,
)

print("=== ETAPA 1: Ganhos Contínuos Projetados ===")
print("G0:\n", cont_design["G0"])
print("G1:\n", cont_design["G1"])
print("G2:\n", cont_design["G2"])
print("L0 (Domínio Original):\n", cont_design["L0"])
print("L2 (Domínio Original):\n", cont_design["L2"])
print("Validação Espectral:", cont_design["spectral_check"])

# 2. Injeção direta no resolvedor de LMIs Multimodo (Etapa 2)
observer_params = {
    "A": A,
    "C": C,
    "h": h,
    "nu_bar": 1000 * h,
    "lambda_obs": 1e-2,
    "L0": cont_design["L0"],
    "L2": cont_design["L2"],
}

obs_result = ControllerDesigner.synthesize_impulsive_observer(
    observer_params, q_min=1.0)
if obs_result is not None:
  print("\n=== ETAPA 2: Ganho de Salto Impulsivo (L1) ===")
  print("L1:\n", obs_result["L1"])
  print(
      f"Pior raio espectral discreto em nu_bar: {obs_result['worst_case']['spectral_radius']:.4f}"
  )
else:
  print("\n=== ETAPA 2: Síntese do Observador Impulsivo Falhou ===")

=== ETAPA 1: Ganhos Contínuos Projetados ===
G0:
 [[-1.18653846]]
G1:
 [[5.08653902]]
G2:
 [[-4.3442847]]
L0 (Domínio Original):
 [[-1.14090237  0.22818047]
 [ 0.22818047 -0.04563609]]
L2 (Domínio Original):
 [[ 8.13935971]
 [10.37212918]]
Validação Espectral: {'max_real_A22_plus_G0': -2.1, 'max_real_A_z': -3.0000002807777646, 'max_real_A_e': 1.9364916731037085, 'is_A22_stable': True, 'is_Az_stable': True}

=== ETAPA 2: Ganho de Salto Impulsivo (L1) ===
L1:
 [[0.93894571]
 [1.78297279]]
Pior raio espectral discreto em nu_bar: 0.9970


In [130]:
import numpy as np


def format_matrix_to_cpp(mat, name="X", scientific=True):
  """Converte um vetor ou matriz numpy para o formato C++ com chave única:

  X = {val1, val2, val3, ...};
  """
  mat = np.atleast_2d(mat)
  flat_vals = mat.flatten()

  fmt = "{:.2e}" if scientific else "{:.4f}"
  vals_str = ", ".join([fmt.format(val) for val in flat_vals])

  return f"{name} = {{{vals_str}}};"


# Extração das variáveis do novo escopo Dual-Channel
Ξ_sc = synth_res["etm"]["sc"]["Ξ"]
Ψ_sc = synth_res["etm"]["sc"]["Ψ"]
Ξ_ca = synth_res["etm"]["ca"]["Ξ"]
Ψ_ca = synth_res["etm"]["ca"]["Ψ"]
K = synth_res["controller"]["K"]
L0 = obs_result["L0"]
L1 = obs_result["L1"]
L2 = obs_result["L2"]

# Exibição no console / Jupyter no formato exato solicitado
print(format_matrix_to_cpp(Ξ_sc, name="Ξ_sc"))
print(format_matrix_to_cpp(Ψ_sc, name="Ψ_sc"))
print(format_matrix_to_cpp(Ξ_ca, name="Ξ_ca"))
print(format_matrix_to_cpp(Ψ_ca, name="Ψ_ca"))
print(format_matrix_to_cpp(K, name="K"))
print(format_matrix_to_cpp(L0, name="L0"))
print(format_matrix_to_cpp(L1, name="L1"))
print(format_matrix_to_cpp(L2, name="L2"))

Ξ_sc = {1.06e+01};
Ψ_sc = {9.16e-02};
Ξ_ca = {2.95e+06, -1.26e+06, -1.26e+06, 5.38e+05};
Ψ_ca = {9.23e+04, -2.91e+04, -2.91e+04, 1.19e+04};
K = {-2.52e+01, -1.85e+01};
L0 = {-1.14e+00, 2.28e-01, 2.28e-01, -4.56e-02};
L1 = {9.39e-01, 1.78e+00};
L2 = {8.14e+00, 1.04e+01};
